# 1322. Ads Performance

## Problem Description

An advertising log records a user's action on an advertisement: Clicked, Viewed, or Ignored.

### Table Schema

**Ads**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| ad_id | int | advertisement identifier |
| user_id | int | user identifier |
| action | enum | Clicked, Viewed or Ignored |

### Task

The pair (ad_id,user_id) is unique. For each advertisement, define CTR as 100 × Clicked / (Clicked + Viewed). Ignored rows are excluded from both numerator and denominator. If there are no clicks or views, return CTR = 0. Round to two decimal places. Return ad_id, ctr for every advertisement, ordered by ctr descending and then ad_id ascending to break ties.

## Example

### Input

**ads**

| ad_id | user_id | action |
| --- | --- | --- |
| 1 | 1 | Clicked |
| 1 | 2 | Viewed |
| 1 | 3 | Ignored |
| 2 | 1 | Ignored |
| 3 | 1 | Viewed |
| 4 | 1 | Clicked |
| 4 | 2 | Clicked |
| 4 | 3 | Viewed |

### Expected Output

| ad_id | ctr |
| --- | --- |
| 4 | 66.67 |
| 1 | 50 |
| 2 | 0 |
| 3 | 0 |

### Explanation

Ad 4 has two clicks and one view, giving 66.67. Ad 1 has one click and one view, giving 50.00; its ignored row does not affect CTR. Ads 2 and 3 have CTR 0 and are ordered by ID.

## Practice

The next code cell constructs exactly these example DataFrames. Try the blank practice cell before reading the solutions.

## References

The problem is restated for self-contained practice; the examples are original. Community links are for further reading, and implementations below are independently written.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/1300-1399/1322.Ads%20Performance/README_EN.md) · [LeetCode](https://leetcode.com/problems/ads-performance/)

[LeetCode discussions](https://leetcode.com/problems/ads-performance/solutions/)

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal
from decimal import Decimal, ROUND_HALF_UP

def percent(n,d):
    return float((Decimal(int(n))*100/Decimal(int(d))).quantize(Decimal('0.01'),rounding=ROUND_HALF_UP)) if d else None

ads = pd.DataFrame([[1,1,"Clicked"],[1,2,"Viewed"],[1,3,"Ignored"],[2,1,"Ignored"],[3,1,"Viewed"],[4,1,"Clicked"],[4,2,"Clicked"],[4,3,"Viewed"]], columns=["ad_id","user_id","action"])
expected = pd.DataFrame([[4,66.67],[1,50],[2,0],[3,0]], columns=["ad_id","ctr"])
display(ads)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Crosstab
crosstab 生成頻數；補上可能不存在的 action。

In [ ]:
def solution_1(ads):
    g = pd.crosstab(ads.ad_id,ads.action).reindex(columns=['Clicked','Viewed'],fill_value=0)
    g['ctr'] = g.apply(lambda r: percent(r.Clicked,r.Clicked+r.Viewed) if r.Clicked+r.Viewed else 0,axis=1)
    return g.reset_index()[['ad_id','ctr']].sort_values(['ctr','ad_id'],ascending=[False,True])

result_1 = solution_1(ads.copy())
display(result_1)

### Solution 2 — Conditional aggregation
分母排除 Ignored；所有 ads 仍保留。

In [ ]:
def solution_2(ads):
    d = ads.assign(click=ads.action.eq('Clicked'),eligible=ads.action.isin(['Clicked','Viewed']))
    g = d.groupby('ad_id',as_index=False).agg(click=('click','sum'),eligible=('eligible','sum'))
    g['ctr'] = g.apply(lambda r: percent(r.click,r.eligible) if r.eligible else 0,axis=1)
    return g[['ad_id','ctr']].sort_values(['ctr','ad_id'],ascending=[False,True])

result_2 = solution_2(ads.copy())
display(result_2)

### Solution 3 — Filtered counts and reindex
分開聚合 numerator/denominator，再補齊零分母群組。

In [ ]:
def solution_3(ads):
    g = ads.loc[ads.action.isin(['Clicked','Viewed'])].groupby('ad_id').size().rename('eligible')
    c = ads.loc[ads.action.eq('Clicked')].groupby('ad_id').size().rename('click')
    d = pd.concat([g,c],axis=1).reindex(pd.Index(ads.ad_id.unique(),name='ad_id')).fillna(0)
    d['ctr'] = d.apply(lambda r: percent(r.click,r.eligible) if r.eligible else 0,axis=1)
    return d.reset_index()[['ad_id','ctr']].sort_values(['ctr','ad_id'],ascending=[False,True])

result_3 = solution_3(ads.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare row grain, missing matches, duplicate keys and zero-count categories.